# `derived_8.4-ece-model-salvage-1.1`

This report evaluates six retrained routing/model families at 60 selected features and the global model at 40, 50, 60, and 69 selected features after removing every SMAP-derived input. All selector, router, and model fitting is restricted to the seven Washington training stations; ECE targets are held out for spatial evaluation only.

## Reproducible setup

The next cell loads the tracked runner and generated artifacts so the notebook remains a reporting layer rather than a second implementation of the experiment.

In [ ]:
from pathlib import Path
import importlib.util
import json
import sys
import pandas as pd

EXP_DIR = Path("experiment/derived_8.4-ece-model-salvage-1.1").resolve()
if not (EXP_DIR / "run_model_salvage.py").exists():
    EXP_DIR = Path.cwd().resolve()
spec = importlib.util.spec_from_file_location("model_salvage_11", EXP_DIR / "run_model_salvage.py")
runner = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = runner
spec.loader.exec_module(runner)
config = runner.load_configuration()
predictions = pd.read_csv(EXP_DIR / "predictions.csv", low_memory=False)
seed_metrics = pd.read_csv(EXP_DIR / "seed_metrics.csv", low_memory=False)
summary = pd.read_csv(EXP_DIR / "summary.csv", low_memory=False)
audit = pd.read_csv(EXP_DIR / "routing_audit.csv", low_memory=False)
print(f"Experiment: {config['experiment']['name']}")
print(f"Models: {len(config['models'])}; feature sizes: {config['selected_feature_sizes']}; seeds: {config['seeds']}")
print(f"Prediction rows: {len(predictions):,}")

## Feature-selection provenance

The local selector is a copied and adapted MI → ElasticNet → stability → wrapper pipeline. Its accepted wrapper objective uses the WA 2023–2025 evaluation period as a documented manual-selection analogue. One evidence-ranked wrapper result is normalized into nested 40/50/60/69 feature manifests, with no specialist or delta additions.

In [ ]:
print("REPORT_BEGIN::SELECTION")
selection_path = EXP_DIR / "feature_selection_artifacts" / "selected_features_by_size.json"
selection = json.loads(selection_path.read_text(encoding="utf-8"))
selected_by_size = selection["feature_sizes"]
candidate_pool = pd.read_csv(EXP_DIR / "feature_selection_artifacts" / "candidate_pool.csv")
selection_table = pd.DataFrame([
    {
        "status": selection.get("status"),
        "selected_count": int(size),
        "smap_selected": sum("smap" in feature.lower() for feature in record["features"]),
        "candidate_pool_count": len(candidate_pool),
        "delta_additions": selection.get("delta_additions", {}),
        "selection_period": "WA 2023–2025",
        "fit_scope": "WA only",
    }
    for size, record in sorted(selected_by_size.items(), key=lambda item: int(item[0]))
])
print(selection_table.to_markdown(index=False))
for size, record in sorted(selected_by_size.items(), key=lambda item: int(item[0])):
    print(f"Selected {size} features:", ";".join(record["features"]))
print("REPORT_END::SELECTION")

## Input and feature audit

This audit records the exact training/evaluation populations, the nested selected feature manifests, and the lineage-specific router inputs after SMAP filtering.

In [ ]:
print("REPORT_BEGIN::INPUT_AUDIT")
with (EXP_DIR / "input_audit.json").open(encoding="utf-8") as handle:
    print(json.dumps(json.load(handle), indent=2, sort_keys=True))
print("REPORT_END::INPUT_AUDIT")

print("REPORT_BEGIN::FEATURE_AUDIT")
with (EXP_DIR / "feature_manifest.json").open(encoding="utf-8") as handle:
    feature_manifest = json.load(handle)
feature_rows = []
for name, value in feature_manifest.items():
    if name == "selected_model_feature_sets":
        for size, selected_value in sorted(value.items(), key=lambda item: int(item[0])):
            feature_rows.append({
                "component": f"selected_model_{size}",
                "parent_count": "selector",
                "dropped_smap": 0,
                "effective_count": len(selected_value["effective"]),
                "effective_features": ";".join(selected_value["effective"]),
            })
    elif isinstance(value, dict) and "effective" in value:
        feature_rows.append({
            "component": name,
            "parent_count": len(value["parent"]) if isinstance(value["parent"], list) else value["parent"],
            "dropped_smap": len(value["dropped"]) if isinstance(value["dropped"], list) else 0,
            "effective_count": len(value["effective"]),
            "effective_features": ";".join(value["effective"]),
        })
feature_table = pd.DataFrame(feature_rows)
print(feature_table.to_markdown(index=False))
print("REPORT_END::FEATURE_AUDIT")

## Router audit

Regime shares are shown for WA trainval, the held-out WA temporal test, and the unseen ECE sensor set. Router seed remains fixed at 42.

In [ ]:
print("REPORT_BEGIN::ROUTER_AUDIT")
print(audit.to_markdown(index=False, floatfmt=".4f"))
print("REPORT_END::ROUTER_AUDIT")

## Ten-variant metrics

RMSE is the primary error metric; Pearson and first-difference Pearson correlation assess whether predictions follow the target level and temporal trend.

In [ ]:
pooled = summary[summary["scope"] == "__pooled__"].copy()
columns = ["model_id", "dataset", "window", "n_seeds", "rmse_mean", "rmse_std", "mae_mean", "bias_mean", "ubrmse_mean", "r2_mean", "pearson_mean", "target_std_mean", "prediction_std_mean", "diff_pearson_mean"]
print("REPORT_BEGIN::METRICS")
print(pooled[columns].sort_values(["dataset", "window", "rmse_mean"]).to_markdown(index=False, floatfmt=".6f"))
print("REPORT_END::METRICS")

## Comparison with original SMAP-trained models

The original formal-evaluation results remain reference-only and are never used for 1.1 fitting.

In [ ]:
comparison = pd.read_csv(EXP_DIR / "reference_comparison.csv", low_memory=False)
print("REPORT_BEGIN::REFERENCE_COMPARISON")
if comparison.empty:
    print("No reference rows available.")
else:
    columns = ["model_id", "seed", "dataset", "rmse_no_smap", "rmse_original", "rmse_delta_no_smap_minus_original", "pearson_no_smap", "pearson_original"]
    print(comparison[columns].to_markdown(index=False, floatfmt=".6f"))
print("REPORT_END::REFERENCE_COMPARISON")

## Comparison of 1.1 selected features with 1.0 no-SMAP models

This same-seed comparison isolates the feature-selection change from the original SMAP-removal change. It is populated when the completed 1.0 prediction artifacts are available.

In [ ]:
comparison_10 = pd.read_csv(EXP_DIR / "salvage_1_1_vs_1_0_summary.csv", low_memory=False)
print("REPORT_BEGIN::SALVAGE_1_1_VS_1_0")
if comparison_10.empty:
    print("No completed 1.0 paired prediction artifacts available yet.")
else:
    columns = ["model_id", "old_model_id", "dataset", "window", "n_seeds", "change_rmse_mean", "change_mae_mean", "change_bias_mean", "change_pearson_mean", "change_diff_pearson_mean"]
    print(comparison_10[columns].sort_values(["dataset", "window", "change_rmse_mean"]).to_markdown(index=False, floatfmt=".6f"))
print("REPORT_END::SALVAGE_1_1_VS_1_0")

## Effect of Removing SMAP: ECE Benefit vs WA Degradation

This paired summary compares 1.1 against the original SMAP-trained references. ECE benefit is original RMSE minus no-SMAP RMSE; WA degradation is no-SMAP RMSE minus original RMSE. Global feature-count variants are listed separately.

In [ ]:
effect_summary = pd.read_csv(EXP_DIR / "old_vs_new_effect_summary.csv", low_memory=False)
effect_columns = ["model_id", "split", "n_seeds", "rmse_original_mean", "rmse_no_smap_mean", "effect_rmse_mean", "effect_rmse_std", "effect_rmse_pct_mean", "improved_seeds", "worsened_seeds", "pearson_change_mean", "diff_pearson_change_mean"]
print("REPORT_BEGIN::OLD_NEW_EFFECT")
if effect_summary.empty:
    print("No paired old-vs-new rows available.")
else:
    print(effect_summary[effect_columns].sort_values(["split", "effect_rmse_mean"], ascending=[True, False]).to_markdown(index=False, floatfmt=".6f"))
effect_figure = runner.make_old_vs_new_effect_figure(effect_summary, EXP_DIR / "figures")
print(f"FIGURE::{effect_figure.name}")
print("REPORT_END::OLD_NEW_EFFECT")

## SMAP-invariance check

Seed-42 ECE rows are evaluated once with native SMAP values and once with all SMAP columns replaced by zero; a no-SMAP model must produce identical regimes and predictions.

In [ ]:
invariance = pd.read_csv(EXP_DIR / "smap_invariance.csv", low_memory=False)
print("REPORT_BEGIN::SMAP_INVARIANCE")
print(invariance.to_markdown(index=False, floatfmt=".12f"))
print("REPORT_END::SMAP_INVARIANCE")

## Global model version comparison

Each ECE station receives one seven-line chart comparing the original global model, the 1.0 no-SMAP global model, the 1.1 global models at 40/50/60/69 features, and ground truth. Predictions are aligned by station/date and averaged over the common seeds `[42, 7, 13]`.

In [ ]:
print("REPORT_BEGIN::GLOBAL_VERSION")
version_data = runner.load_data(config)
version_status = runner.global_version_source_status(version_data, config, (42, 7, 13))
print(json.dumps(version_status, indent=2, sort_keys=True))
if version_status["ready"]:
    version_paths = runner.make_global_version_charts(version_data, config, EXP_DIR / "figures", (42, 7, 13))
    for path in version_paths:
        print(f"FIGURE::{path.name}")
else:
    print("Global-version figures deferred until all old, 1.0, and 1.1 common-seed artifacts are complete.")
print("REPORT_END::GLOBAL_VERSION")

## Trend figures

The next cell generates three figure suites per ECE station: architecture gates, alternative regime gates, and all four global feature sizes. Each chart is limited to five lines including the observed target.

In [ ]:
figure_paths = runner.make_trend_figures(predictions, EXP_DIR / "figures")
print("REPORT_BEGIN::FIGURES")
for path in figure_paths:
    print(f"- {path.name}")
print("REPORT_END::FIGURES")

## Completion

All tables above are sourced from executed cells, and the linked figures are generated during this notebook execution.

In [ ]:
print("Notebook complete — all report sections above are generated from experiment artifacts.")